# Лекція 10: Міграції, зв'язки та цілісність даних

**Курс:** Прикладна розробка ПЗ на Python 2026

**Передумови:** Лекція 9 (Docker, PostgreSQL, SQLAlchemy, repository pattern)

## Цілі лекції

Після цієї лекції ви зможете:

1. Пояснити, навіщо потрібні міграції бази даних і чим вони відрізняються від `create_all()`
2. Використовувати Alembic для створення, застосування та відкату міграцій
3. Проєктувати зв'язки між таблицями (one-to-many) з використанням `ForeignKey` та `relationship()`
4. Писати тести, що працюють із реляційними моделями
5. Діагностувати стан бази даних через `psql`

## Вступ — навіщо міграції?

### Проблема

Уявіть ситуацію: ви змінили модель (додали нове поле, видалили старе), але в базі даних вже є реальні дані користувачів.

Що робить `Base.metadata.create_all()`?

```python
# app/main.py (Лекція 9)
@asynccontextmanager
async def lifespan(app: FastAPI):
    Base.metadata.create_all(bind=engine)  # Створює таблиці, якщо їх немає
    yield
```

**Проблема:** `create_all()` **не оновлює** існуючі таблиці. Якщо таблиця вже існує — нові колонки НЕ з'являться. А якщо ви спочатку зробите `drop_all()` + `create_all()` — всі дані будуть **втрачені назавжди**.

![Мем про втрату даних](https://http.cat/images/410.jpg)

*«Я просто змінив модель... де мої дані?»*

### Рішення: міграції

**Міграції** — це версійний контроль для схеми бази даних. Кожна міграція описує конкретну зміну: додати колонку, створити таблицю, змінити тип даних. Можна рухатися вперед (`upgrade`) і назад (`downgrade`).

| `create_all()` | Міграції (Alembic) |
|---|---|
| Створює таблиці з нуля | Поступово змінює схему |
| Не змінює існуючі таблиці | Точно описує кожну зміну |
| Дані втрачаються при drop+create | Дані зберігаються |
| Ніякої історії | Повна історія змін |
| Добре для тестів | Добре для production |

## Alembic крок за кроком

**Alembic** — це інструмент міграцій для SQLAlchemy. Він автоматично визначає різницю між вашими моделями та поточним станом бази.

### Крок 1: Ініціалізація

```bash
# В директорії проєкту
alembic init alembic
```

Це створює структуру:

```
notes-api/
├── alembic/
│   ├── env.py          # Конфігурація: звідки брати моделі та URL бази
│   ├── script.py.mako  # Шаблон для нових міграцій
│   └── versions/       # Файли міграцій
├── alembic.ini          # Головний конфіг
└── app/
```

### Крок 2: Налаштування `env.py`

Alembic має знати про ваші моделі та як підключитися до бази. Дивіться наступну клітинку.

### Крок 3: Генерація міграції

```bash
alembic revision --autogenerate -m "initial notes table"
```

Alembic порівнює моделі з поточним станом бази і генерує файл міграції з операціями `upgrade()` та `downgrade()`.

### Крок 4: Застосування

```bash
alembic upgrade head    # Застосувати всі міграції до останньої
alembic upgrade +1      # Застосувати наступну міграцію
```

### Крок 5: Відкат

```bash
alembic downgrade -1    # Відкотити одну міграцію назад
alembic downgrade base  # Повернутися до початкового стану
```

### Крок 6: Перевірка поточної версії

```bash
alembic current         # Яка версія зараз?
alembic history         # Повна історія міграцій
```

### Важливо: autogenerate не ідеальний!

`--autogenerate` — це допомога, а **не заміна** ручної перевірки. Alembic **не може** виявити:

- Перейменування таблиць або колонок (бачить як "видалити + створити")
- Зміни в даних (data migrations)
- CHECK-обмеження
- Зміни типів у деяких випадках

> ⚠️ **Правило**: завжди переглядайте згенерований файл міграції перед `alembic upgrade`. Одна неперевірена міграція може знищити дані.

### Перехід від `create_all()` до міграцій

У Лекції 9 наш `app/main.py` мав такий код:

```python
# Лекція 9 — старий підхід
@asynccontextmanager
async def lifespan(app: FastAPI):
    Base.metadata.create_all(bind=engine)  # створює таблиці при старті
    yield
```

Тепер, коли Alembic керує схемою, ми **видаляємо** `create_all()` з `main.py`:

```python
# Лекція 10 — новий підхід (без create_all)
app = FastAPI(title=settings.app_name, version="0.3.0", debug=settings.debug)
# Ніякого lifespan — Alembic керує схемою
```

**Чому?** Якщо `create_all()` і Alembic працюють одночасно:
- `create_all()` створить таблиці, але Alembic не знатиме про це
- `alembic upgrade head` спробує створити таблиці знову → помилка
- Історія міграцій буде неконсистентною

> 💡 **Якщо у вас вже є база даних з Лекції 9** (створена через `create_all()`), виконайте:
> ```bash
> alembic stamp head
> ```
> Це позначить поточний стан бази як "всі міграції застосовані" без фактичного виконання міграцій.

In [ ]:
# alembic/env.py — конфігурація Alembic для нашого проєкту

import os
from logging.config import fileConfig

from sqlalchemy import engine_from_config, pool

from alembic import context

from app.database import Base
from app.models import note  # noqa: F401
from app.models import tag   # noqa: F401

config = context.config

if config.config_file_name is not None:
    fileConfig(config.config_file_name)

# Ключовий момент: URL бази з змінної середовища
database_url = os.environ.get("DATABASE_URL")
if database_url:
    config.set_main_option("sqlalchemy.url", database_url)

# Alembic порівнює моделі з БД через metadata
target_metadata = Base.metadata


def run_migrations_offline() -> None:
    url = config.get_main_option("sqlalchemy.url")
    context.configure(
        url=url,
        target_metadata=target_metadata,
        literal_binds=True,
        dialect_opts={"paramstyle": "named"},
    )
    with context.begin_transaction():
        context.run_migrations()


def run_migrations_online() -> None:
    connectable = engine_from_config(
        config.get_section(config.config_ini_section, {}),
        prefix="sqlalchemy.",
        poolclass=pool.NullPool,
    )
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=target_metadata)
        with context.begin_transaction():
            context.run_migrations()


if context.is_offline_mode():
    run_migrations_offline()
else:
    run_migrations_online()

In [ ]:
# alembic/versions/14458ff7b2b5_initial_notes_table.py
# Перша міграція — створення таблиці notes

"""initial notes table

Revision ID: 14458ff7b2b5
Revises:
Create Date: 2026-04-09 14:37:02.459044
"""

from typing import Sequence, Union
from alembic import op
import sqlalchemy as sa

revision: str = "14458ff7b2b5"
down_revision: Union[str, Sequence[str], None] = None  # Перша міграція — немає попередньої


def upgrade() -> None:
    """Upgrade schema."""
    op.create_table(
        "notes",
        sa.Column("id", sa.String(length=36), nullable=False),
        sa.Column("title", sa.String(length=200), nullable=False),
        sa.Column("content", sa.Text(), nullable=False),
        sa.Column("tags", sa.JSON(), nullable=False),        # <-- JSON колонка (тимчасово)
        sa.Column("created_at", sa.DateTime(timezone=True), nullable=False),
        sa.PrimaryKeyConstraint("id"),
    )


def downgrade() -> None:
    """Downgrade schema."""
    op.drop_table("notes")

## Вправа 1: Перевірка міграцій

Переконайтесь, що Docker-контейнер PostgreSQL запущений, і виконайте:

```bash
# 1. Застосувати всі міграції
alembic upgrade head

# 2. Підключитися до бази через psql
docker exec -it notes-db psql -U notes_user -d notes_db

# 3. В psql перевірити таблиці
\dt

# 4. Подивитися структуру таблиці notes
\d notes

# 5. Перевірити таблицю версій Alembic
SELECT * FROM alembic_version;
```

**Очікуваний результат:**
- Таблиця `notes` з колонками `id`, `title`, `content`, `tags` (JSON), `created_at`
- Таблиця `alembic_version` з поточним revision ID

<details>
<summary><b>Розв'язок вправи 1</b> (натисніть, щоб розгорнути)</summary>

```
$ alembic upgrade head
INFO  [alembic.runtime.migration] Running upgrade -> 14458ff7b2b5, initial notes table
```

```
$ docker compose exec db psql -U notes -d notes_db

notes_db=# \dt
              List of relations
 Schema |      Name       | Type  | Owner
--------+-----------------+-------+-------
 public | alembic_version | table | notes
 public | notes           | table | notes

notes_db=# \d notes
     Column    |           Type           | Nullable
---------------+--------------------------+----------
 id            | character varying(36)    | not null
 title         | character varying(200)   | not null
 content       | text                     | not null
 tags          | json                     |          
 created_at    | timestamp with time zone | not null
```

> Зверніть увагу: поки що таблиця `notes` має колонку `tags` типу JSON — це стан до рефакторингу на зв'язки.

</details>

## Зв'язки між таблицями

### Чому JSON-колонка — погана ідея?

У першій версії нашої моделі теги зберігалися як JSON-масив прямо в таблиці `notes`:

```python
tags: Mapped[list] = mapped_column(JSON, default=list)
```

**Проблеми:**
- Неможливо побудувати індекс на окремому тегу
- Неможливо зробити `JOIN` або ефективний пошук "всі нотатки з тегом X"
- Неможливо задати обмеження (constraints) на окремі значення
- Немає referential integrity

### Рішення: зв'язок один-до-багатьох (one-to-many)

Виносимо теги в окрему таблицю зі зв'язком через зовнішній ключ (foreign key):

```
┌──────────────────┐         ┌──────────────────┐
│      notes       │         │       tags       │
├──────────────────┤         ├──────────────────┤
│ id (PK)     ◄────┼─────────┤ note_id (FK)     │
│ title            │    1:N  │ id (PK)          │
│ content          │         │ name             │
│ created_at       │         └──────────────────┘
└──────────────────┘
```

Одна нотатка може мати **багато** тегів. Кожен тег належить **одній** нотатці.

### Ключові концепції

- **`ForeignKey("notes.id")`** — зовнішній ключ на рівні бази даних. БД гарантує, що `note_id` посилається на існуючий запис.
- **`relationship()`** — зв'язок на рівні Python/SQLAlchemy. Дозволяє звертатися до `note.tags` як до списку об'єктів.
- **`back_populates`** — двосторонній зв'язок: `note.tags` та `tag.note` синхронізовані.
- **`cascade="all, delete-orphan"`** — при видаленні нотатки автоматично видаляються всі її теги.
- **`ondelete="CASCADE"`** — каскадне видалення на рівні бази даних (додатковий захист).

### Чому Tag використовує `int`, а Note — `str` (UUID)?

| Тип ID | Переваги | Недоліки | Коли використовувати |
|--------|----------|----------|---------------------|
| `int` (autoincrement) | Простий, швидкий, компактний | Передбачуваний (можна вгадати ID) | Внутрішні сутності (теги, категорії) |
| `str` (UUID) | Унікальний глобально, безпечний | Довший, повільніший для індексів | Публічні ресурси (нотатки, користувачі) |

Tag — це **внутрішня** сутність, яку клієнт API не адресує напряму. Note — це **публічний** ресурс з ID в URL. Тому Note використовує UUID (не можна вгадати чужу нотатку), а Tag — простий integer.

In [ ]:
# app/models/tag.py — модель тегу

from __future__ import annotations

from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import Mapped, mapped_column, relationship

from app.database import Base


class TagModel(Base):
    __tablename__ = "tags"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(50), nullable=False)
    note_id: Mapped[str] = mapped_column(
        ForeignKey("notes.id", ondelete="CASCADE"), nullable=False
    )

    note: Mapped["NoteModel"] = relationship(back_populates="tags")

In [ ]:
# app/models/note.py — оновлена модель нотатки (без JSON tags)

from __future__ import annotations

import uuid
from datetime import UTC, datetime

from sqlalchemy import DateTime, String, Text
from sqlalchemy.orm import Mapped, mapped_column, relationship

from app.database import Base


class NoteModel(Base):
    __tablename__ = "notes"

    id: Mapped[str] = mapped_column(
        String(36), primary_key=True, default=lambda: str(uuid.uuid4())
    )
    title: Mapped[str] = mapped_column(String(200), nullable=False)
    content: Mapped[str] = mapped_column(Text, nullable=False)
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), default=lambda: datetime.now(UTC)
    )

    # Зв'язок з тегами: один-до-багатьох
    # cascade="all, delete-orphan" — видалення нотатки видаляє всі теги
    tags: Mapped[list["TagModel"]] = relationship(
        back_populates="note", cascade="all, delete-orphan"
    )

## Міграція для зв'язку

Після того, як ми створили `TagModel` та видалили JSON-колонку з `NoteModel`, потрібна нова міграція:

```bash
alembic revision --autogenerate -m "add tags table"
```

Alembic автоматично визначив:
1. **Нова таблиця** `tags` з колонками `id`, `name`, `note_id` та зовнішнім ключем
2. **Видалення колонки** `tags` (JSON) з таблиці `notes`

Зверніть увагу на `down_revision` — вона вказує на попередню міграцію. Це створює ланцюжок:

```
(порожня БД) → 14458ff7b2b5 (initial) → 1f835e342a17 (add tags)
```

### Що трапиться при порушенні зовнішнього ключа?

Якщо спробувати створити тег з неіснуючим `note_id`, PostgreSQL поверне помилку:

```
IntegrityError: insert or update on table "tags" violates
foreign key constraint "tags_note_id_fkey"
DETAIL: Key (note_id)=(nonexistent) is not present in table "notes".
```

Це і є **цілісність даних** (data integrity) — база даних **не дозволить** створити "осиротілий" тег без нотатки.

У нашому коді це обробляється через `cascade="all, delete-orphan"`:
- Видалення нотатки → автоматичне видалення всіх її тегів
- Спроба додати тег до неіснуючої нотатки → помилка

**Чому потрібні ОБА рівні захисту?**

| Рівень | Механізм | Що робить |
|--------|----------|-----------|
| Python (ORM) | `cascade="all, delete-orphan"` | SQLAlchemy видаляє теги перед видаленням нотатки |
| SQL (БД) | `ondelete="CASCADE"` | PostgreSQL видалить теги, навіть якщо хтось видалить нотатку напряму через SQL |

> 💡 ORM-каскад працює лише через Python-код. SQL-каскад працює **завжди** — навіть при прямому `DELETE FROM notes` в psql.

In [ ]:
# alembic/versions/1f835e342a17_add_tags_table.py
# Друга міграція — таблиця tags + видалення JSON-колонки

"""add tags table

Revision ID: 1f835e342a17
Revises: 14458ff7b2b5
Create Date: 2026-04-09 14:38:27.405429
"""

from typing import Sequence, Union
from alembic import op
import sqlalchemy as sa

revision: str = "1f835e342a17"
down_revision: Union[str, Sequence[str], None] = "14458ff7b2b5"  # Посилання на попередню


def upgrade() -> None:
    """Upgrade schema."""
    # Створення таблиці tags з зовнішнім ключем
    op.create_table(
        "tags",
        sa.Column("id", sa.Integer(), autoincrement=True, nullable=False),
        sa.Column("name", sa.String(length=50), nullable=False),
        sa.Column("note_id", sa.String(length=36), nullable=False),
        sa.ForeignKeyConstraint(["note_id"], ["notes.id"], ondelete="CASCADE"),
        sa.PrimaryKeyConstraint("id"),
    )
    # Видалення старої JSON-колонки
    op.drop_column("notes", "tags")


def downgrade() -> None:
    """Downgrade schema."""
    op.add_column("notes", sa.Column("tags", sa.JSON(), nullable=False))
    op.drop_table("tags")

## Оновлення коду

Тепер, коли теги живуть в окремій таблиці, потрібно оновити repository та service.

### Що змінилося в repository

**Було (JSON-підхід):**
```python
note = NoteModel(
    title=note_data.title,
    content=note_data.content,
    tags=note_data.tags,  # Просто список рядків у JSON
)
```

**Стало (реляційний підхід):**
```python
note = NoteModel(
    title=note_data.title,
    content=note_data.content,
)
for tag_name in note_data.tags:
    note.tags.append(TagModel(name=tag_name))  # Створюємо об'єкти TagModel
```

### Що змінилося в пошуку

**Було:** неможливо ефективно шукати по тегах в JSON

**Стало:** `JOIN` + `WHERE IN`
```python
stmt = stmt.join(NoteModel.tags).where(TagModel.name.in_(tags))
```

### Що змінилося в service

Додалася функція `_note_to_response()` — вона конвертує `TagModel` об'єкти назад у список рядків для API:

```python
tags=[t.name for t in note.tags]  # TagModel → str
```

API залишається тим самим: клієнт надсилає і отримує `tags: ["python", "fastapi"]`. Зміна прозора.

### Проблема N+1 запитів та `selectinload`

Коли ви завантажуєте нотатку з бази, SQLAlchemy **не завантажує** пов'язані теги автоматично. Це називається **ліниве завантаження** (lazy loading):

```python
# Без selectinload — КОЖЕН доступ до note.tags робить окремий SQL-запит
notes = db.scalars(select(NoteModel)).all()  # 1 запит
for note in notes:
    print(note.tags)  # +1 запит для КОЖНОЇ нотатки!
# Якщо 100 нотаток → 101 запит (1 + 100) = N+1 проблема
```

**Рішення** — `selectinload` завантажує всі теги **одним додатковим запитом**:

```python
# З selectinload — лише 2 запити незалежно від кількості нотаток
notes = db.scalars(
    select(NoteModel).options(selectinload(NoteModel.tags))
).all()  # 1 запит для нотаток + 1 запит для ВСІХ тегів = 2 запити
```

| Підхід | Кількість запитів | Коли використовувати |
|--------|:-----------------:|---------------------|
| Lazy loading (за замовчуванням) | 1 + N | Коли теги потрібні рідко |
| `selectinload` | 2 | Коли завжди потрібні пов'язані дані |
| `joinedload` | 1 (з JOIN) | Для простих зв'язків без дублювання |


In [ ]:
# app/repositories/notes.py — оновлений repository

from __future__ import annotations

from sqlalchemy import select
from sqlalchemy.orm import Session, selectinload

from app.models.note import NoteModel
from app.models.tag import TagModel
from app.schemas.notes import NoteCreate


def create_note(db: Session, note_data: NoteCreate) -> NoteModel:
    note = NoteModel(
        title=note_data.title,
        content=note_data.content,
    )
    for tag_name in note_data.tags:
        note.tags.append(TagModel(name=tag_name))
    db.add(note)
    db.commit()
    db.refresh(note)
    return note


def get_note_by_id(db: Session, note_id: str) -> NoteModel | None:
    # selectinload — завантажує теги одним додатковим запитом (уникаємо N+1)
    stmt = (
        select(NoteModel)
        .options(selectinload(NoteModel.tags))
        .where(NoteModel.id == note_id)
    )
    return db.scalars(stmt).first()


def search_notes(
    db: Session,
    query: str = "",
    tags: list[str] | None = None,
    limit: int = 10,
) -> list[NoteModel]:
    stmt = select(NoteModel).options(selectinload(NoteModel.tags))

    if query:
        stmt = stmt.where(
            NoteModel.title.ilike(f"%{query}%") | NoteModel.content.ilike(f"%{query}%")
        )

    if tags:
        # JOIN з таблицею tags для фільтрації
        stmt = stmt.join(NoteModel.tags).where(TagModel.name.in_(tags))

    stmt = stmt.distinct().limit(limit)
    return list(db.scalars(stmt).all())


def delete_note(db: Session, note_id: str) -> bool:
    note = db.get(NoteModel, note_id)
    if note is None:
        return False
    db.delete(note)  # cascade видалить теги автоматично
    db.commit()
    return True

In [ ]:
# app/services/notes.py — оновлений service

from __future__ import annotations

from fastapi import HTTPException, status
from sqlalchemy.orm import Session

from app.models.note import NoteModel
from app.repositories import notes as notes_repo
from app.schemas.notes import (
    NoteCreate,
    NoteResponse,
    NoteSearchQuery,
    NoteSearchResult,
)


def _note_to_response(note: NoteModel) -> NoteResponse:
    """Конвертує ORM-модель у Pydantic-схему для API."""
    return NoteResponse(
        id=note.id,
        title=note.title,
        content=note.content,
        tags=[t.name for t in note.tags],  # TagModel -> str
        created_at=note.created_at,
    )


def create_note(db: Session, note_data: NoteCreate) -> NoteResponse:
    note = notes_repo.create_note(db, note_data)
    return _note_to_response(note)


def get_note(db: Session, note_id: str) -> NoteResponse:
    note = notes_repo.get_note_by_id(db, note_id)
    if note is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Note {note_id} not found",
        )
    return _note_to_response(note)


def search_notes(db: Session, query: NoteSearchQuery) -> NoteSearchResult:
    notes = notes_repo.search_notes(
        db, query=query.query, tags=query.tags, limit=query.limit
    )
    results = [_note_to_response(n) for n in notes]
    return NoteSearchResult(results=results, total=len(results))


def delete_note(db: Session, note_id: str) -> None:
    deleted = notes_repo.delete_note(db, note_id)
    if not deleted:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Note {note_id} not found",
        )

## Вправа 2: Унікальне обмеження для тегів

Зараз нічого не заважає додати один і той самий тег до нотатки двічі. Це логічна помилка.

**Завдання:**

1. Додайте унікальне обмеження (unique constraint) на пару `(name, note_id)` в `TagModel`, щоб один тег не міг повторюватися в межах однієї нотатки.

   Підказка: використайте `UniqueConstraint` в `__table_args__`.

2. Згенеруйте та застосуйте міграцію:

```bash
alembic revision --autogenerate -m "unique tag per note"
alembic upgrade head
```

3. Перевірте в psql, що обмеження створилось:

```sql
\d tags
```

<details>
<summary><b>Розв'язок вправи 2</b> (натисніть, щоб розгорнути)</summary>

Додайте в `TagModel`:

```python
from sqlalchemy import ForeignKey, String, UniqueConstraint
from sqlalchemy.orm import Mapped, mapped_column, relationship

from app.database import Base


class TagModel(Base):
    __tablename__ = "tags"
    __table_args__ = (
        UniqueConstraint("name", "note_id", name="uq_tag_name_per_note"),
    )

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(50), nullable=False)
    note_id: Mapped[str] = mapped_column(
        ForeignKey("notes.id", ondelete="CASCADE"), nullable=False
    )

    note: Mapped["NoteModel"] = relationship(back_populates="tags")
```

Потім:

```bash
$ alembic revision --autogenerate -m "unique tag per note"
INFO  Detected added unique constraint 'uq_tag_name_per_note' on '['name', 'note_id']'

$ alembic upgrade head
INFO  Running upgrade 1f835e342a17 -> ..., unique tag per note
```

Перевірка:

```sql
notes_db=# \d tags
-- Має показати constraint "uq_tag_name_per_note" UNIQUE (name, note_id)

-- Спробуйте вставити дублікат — база не дозволить:
INSERT INTO tags (name, note_id) VALUES ('python', 'some-id');
INSERT INTO tags (name, note_id) VALUES ('python', 'some-id');
-- ERROR: duplicate key value violates unique constraint "uq_tag_name_per_note"
```

</details>

## Принципи проєктування бази даних

### Індекси (indexes)

Індекс прискорює пошук, але сповільнює запис. Додавайте індекси на колонки, по яких часто шукаєте:

```python
# Приклад: індекс на note_id для швидкого пошуку тегів нотатки
note_id: Mapped[str] = mapped_column(
    ForeignKey("notes.id", ondelete="CASCADE"),
    nullable=False,
    index=True,  # <-- індекс
)
```

**Коли додавати індекс:**
- Колонки в `WHERE` та `JOIN`
- Зовнішні ключі (foreign keys) — часто індексуються автоматично
- Колонки для сортування (`ORDER BY`)

**Коли НЕ варто:**
- Таблиці з малою кількістю записів
- Колонки, які рідко використовуються у запитах

### Обмеження (constraints)

| Обмеження | Що робить | Приклад |
|---|---|---|
| `NOT NULL` | Значення обов'язкове | `title` нотатки |
| `UNIQUE` | Значення не повторюється | Email користувача |
| `PRIMARY KEY` | Унікальний ідентифікатор рядка | `id` |
| `FOREIGN KEY` | Посилання на інший запис | `note_id` в тегах |
| `CHECK` | Довільна умова | `length > 0` |

### Головний принцип

> **Думайте про запити перш ніж проєктувати таблиці.**
>
> Які питання буде ставити ваш код до бази? "Знайди всі нотатки з тегом X", "Порахуй нотатки за сьогодні" — від цього залежить структура таблиць та індекси.

### Приклад: проєктування під запити

**Задача**: знайти всі нотатки з тегом "important", створені за останній тиждень.

```sql
SELECT n.* FROM notes n
JOIN tags t ON t.note_id = n.id
WHERE t.name = 'important'
  AND n.created_at > NOW() - INTERVAL '7 days';
```

**Які індекси допоможуть?**
1. `tags.name` — для фільтрації тегів за назвою
2. `tags.note_id` — для швидкого JOIN
3. `notes.created_at` — для фільтрації за датою

Без індексів цей запит скануватиме **всі** рядки в обох таблицях.

## Тестування з базою даних

Зверніть увагу: тести використовують `create_all()` замість Alembic-міграцій. Це нормально!

**Чому не Alembic для тестів?**
- `create_all()` працює швидше — створює все за один виклик
- Не потрібно підтримувати порядок міграцій в тестах
- Тестова база створюється з нуля перед кожним тестом і знищується після

**Важливо:** тести використовують SQLite (а не PostgreSQL), тому вони швидкі та не потребують Docker.

Подивіться на `conftest.py` — він імпортує обидві моделі (`NoteModel`, `TagModel`), щоб `create_all()` створив обидві таблиці:

### Порівняння з попередніми підходами

| Лекція | Підхід | База даних | Що тестується |
|--------|--------|:----------:|---------------|
| **L7** | `TestClient` без БД | Немає (заглушки) | HTTP-ендпоінти, валідація |
| **L9** | `TestClient` + SQLite | SQLite (файл) | CRUD через ORM |
| **L10** | `TestClient` + SQLite + FK | SQLite (файл, `PRAGMA foreign_keys=ON`) | CRUD + зв'язки + цілісність |

Ключова різниця: у L7 ендпоінти повертали фейкові дані. Тепер тести перевіряють, що дані **справді зберігаються** в базі через ORM.

> ⚠️ SQLite і PostgreSQL мають відмінності (типи даних, поведінка JSON, обмеження). Для критичних проєктів використовуйте тестовий PostgreSQL-контейнер.

In [ ]:
# tests/conftest.py — тестова конфігурація

from __future__ import annotations

import os

# Встановлюємо тестову БД ПЕРЕД імпортом додатку
os.environ["DATABASE_URL"] = "sqlite:///./test.db"

import pytest
from fastapi.testclient import TestClient
from sqlalchemy import create_engine, event
from sqlalchemy.orm import sessionmaker

from app.database import Base, get_db
from app.main import app
from app.models import NoteModel, TagModel  # noqa: F401 — потрібні для create_all

test_engine = create_engine(
    "sqlite:///./test.db", connect_args={"check_same_thread": False}
)


@event.listens_for(test_engine, "connect")
def _set_sqlite_pragma(dbapi_conn, connection_record):
    cursor = dbapi_conn.cursor()
    cursor.execute("PRAGMA journal_mode=WAL")
    cursor.execute("PRAGMA foreign_keys=ON")  # Забезпечує перевірку FK у тестах
    cursor.close()


TestSessionLocal = sessionmaker(bind=test_engine)


@pytest.fixture
def db_session():
    Base.metadata.create_all(bind=test_engine)  # Створює notes + tags
    session = TestSessionLocal()
    try:
        yield session
    finally:
        session.close()
        Base.metadata.drop_all(bind=test_engine)  # Очищує після тесту


@pytest.fixture
def client(db_session):
    def _get_test_db():
        try:
            yield db_session
        finally:
            pass

    app.dependency_overrides[get_db] = _get_test_db
    with TestClient(app) as c:
        yield c
    app.dependency_overrides.clear()

## psql: швидка діагностика

Набір корисних команд для роботи з PostgreSQL через термінал:

```bash
# Підключення до контейнера
docker exec -it notes-db psql -U notes_user -d notes_db
```

### Основні psql-команди

```sql
-- Список таблиць
\dt

-- Структура таблиці notes
\d notes

-- Структура таблиці tags
\d tags

-- Всі нотатки з тегами (JOIN)
SELECT n.id, n.title, t.name AS tag
FROM notes n
LEFT JOIN tags t ON t.note_id = n.id
ORDER BY n.created_at DESC;

-- Кількість тегів на кожну нотатку
SELECT n.title, COUNT(t.id) AS tag_count
FROM notes n
LEFT JOIN tags t ON t.note_id = n.id
GROUP BY n.id, n.title;

-- Поточна версія міграцій
SELECT * FROM alembic_version;

-- Вихід
\q
```

## Підсумок

- **Міграції** (Alembic) — це версійний контроль для схеми бази даних. Вони дозволяють змінювати структуру без втрати даних.
- **`alembic revision --autogenerate`** порівнює ваші моделі з поточною схемою і генерує файл міграції.
- **Зв'язки** (relationships) дозволяють винести дані в окрему таблицю та працювати з ними як з Python-об'єктами.
- **`ForeignKey`** забезпечує цілісність на рівні бази, **`relationship()`** — зручність на рівні Python.
- **`cascade="all, delete-orphan"`** автоматично видаляє пов'язані записи.
- Тести використовують `create_all()` для швидкості, production — Alembic для безпеки даних.

- **Проєктування БД** — індекси прискорюють читання (але сповільнюють запис), обмеження захищають цілісність даних

![Все працює](https://http.cat/images/200.jpg)

*Коли міграція пройшла успішно з першого разу*

## Що далі?

**Лекція 11:** Аналітика даних з pandas — експорт даних з бази та їх аналіз за допомогою pandas та matplotlib.

## Джерела

1. [Alembic Tutorial](https://alembic.sqlalchemy.org/en/latest/tutorial.html) — офіційний підручник по Alembic
2. [SQLAlchemy Relationships](https://docs.sqlalchemy.org/en/20/orm/relationships.html) — зв'язки між таблицями в SQLAlchemy 2.0
3. [PostgreSQL Indexes](https://www.postgresql.org/docs/current/indexes.html) — документація по індексах PostgreSQL
4. [Alembic Auto Generating Migrations](https://alembic.sqlalchemy.org/en/latest/autogenerate.html) — автогенерація міграцій
5. [Alembic Cookbook](https://alembic.sqlalchemy.org/en/latest/cookbook.html) — рецепти для типових міграційних сценаріїв
6. [SQLAlchemy Loading Strategies](https://docs.sqlalchemy.org/en/20/orm/queryguide/relationships.html) — selectinload, joinedload та інші стратегії завантаження
